<a href="https://colab.research.google.com/github/gocenalper/nano-gpt/blob/main/gpt_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import urllib.request

import torch

In [ ]:
# ---------------------------------------------------------------------------
# Hyperparameter'lar — bu adım için yalnızca veri ile ilgili olanlar.
# ---------------------------------------------------------------------------
BLOCK_SIZE = 128         # context length: aynı anda kaç karakter görür (sonra büyütücez)
BATCH_SIZE = 4         # paralel kaç bağımsız sequence işlenir
TRAIN_FRACTION = 0.9   # %90 train, %10 val
DROPOUT_RATE = 0.2     # Dropout oranı
SEED = 1337            # Karpathy'nin seed'i — birebir aynı sayıları görmek için
_BASE_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
DATA_DIR = _BASE_DIR / "data"
DATA_PATH = DATA_DIR / "input.txt"
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [ ]:
def download_dataset() -> str:
  DATA_DIR.mkdir(exist_ok=True)
  if not DATA_PATH.exists():
    print(f"[download] {DATA_URL} -> {DATA_PATH}")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
  text = DATA_PATH.read_text(encoding="utf-8")
  return text

def build_tokenizer(text: str):
  """
    Character-level tokenizer kur.
    - stoi: string -> int  (her unique karakter bir id)
    - itos: int -> string
    BPE/tiktoken yerine bunu kullanıyoruz çünkü:
      (a) vocab küçük (~65) → embedding tablosu küçük, debugging kolay
      (b) "token bir int'tir, decode geri string verir" özünü çıplak gösterir
    Karşılığı: sequence'lar daha uzun olur (her char bir token).
  """
  chars = sorted(set(text))
  vocab_size = len(chars)
  stoi = {ch: i for i, ch in enumerate(chars)}
  itos = {i: ch for i, ch in enumerate(chars)}

  def encode(s: str) -> list[int]:
      return [stoi[c] for c in s]

  def decode(ids: list[int]) -> str:
      return "".join(itos[i] for i in ids)

  return chars, vocab_size, encode, decode


def make_splits(text: str, encode) -> tuple[torch.Tensor, torch.Tensor]:
    """Tüm metni tek bir int tensoruna çevir, train/val olarak böl."""
    data = torch.tensor(encode(text), dtype=torch.long)
    n = int(TRAIN_FRACTION * len(data))
    return data[:n], data[n:]

def get_batch(split: str, train_data: torch.Tensor, val_data: torch.Tensor,
              block_size: int = BLOCK_SIZE, batch_size: int = BATCH_SIZE,
              device: str = "cpu") -> tuple[torch.Tensor, torch.Tensor]:
    """
    Rastgele bir batch çek.

    Mantık:
      - `batch_size` adet rastgele başlangıç index'i seç
      - Her birinden `block_size` uzunluklu bir x ve onun 1 sağa kaymış hali y al
      - Shape: x, y -> (B, T)

    Neden 1 sağa kaymış? Çünkü hedefimiz "verilen context'in bir sonraki
    token'ı". x[:, t] göz önüne alındığında model y[:, t]'yi tahmin etmeye
    çalışacak. Tek bir block_size'lık dilim, T adet ayrı (context, target)
    eğitim örneği içerir — bu Transformer'ın paralel öğrenme avantajının
    matematiksel kaynağı.
    """
    data = train_data if split == "train" else val_data
    # her örnek için son block_size + 1 elemana yer kalsın diye -block_size
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Now, let's define the neural network architecture. We'll start with the `Head` for self-attention, then `MultiHeadAttention`, `Block`, and finally the `BigramLanguageModel`.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

# ---------------------------------------------------------------------------
# Model tanımı
# ---------------------------------------------------------------------------

class Head(nn.Module):
    """ Bir tane self-attention head """

    def __init__(self, n_embed: int, head_size: int, dropout_rate: float): # n_embed ve dropout_rate eklendi
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.head_size = head_size
        self.dropout = nn.Dropout(dropout_rate) # Dropout katmanı eklendi

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # attention skorlarını hesapla
        wei = q @ k.transpose(-2, -1) * (self.head_size**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei) # Dropout uygulandı

        v = self.value(x)
        out = wei @ v
        return out

In [ ]:
class MultiHeadAttention(nn.Module):
    """ Paralel olarak birden fazla self-attention head çalıştırma """

    def __init__(self, n_embed: int, num_heads: int, head_size: int, dropout_rate: float): # dropout_rate eklendi
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embed, head_size, dropout_rate) for _ in range(num_heads)]) # dropout_rate'i Head'e geçirme
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout_rate) # Dropout katmanı eklendi

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out)) # Projeksiyon ve Dropout uygulandı
        return out

In [ ]:
class Block(nn.Module):
    """ Transformer bloğu: Haberleşme (MultiHeadAttention) + Hesaplama (FFN) """

    def __init__(self, n_embed: int, n_head: int, dropout_rate: float): # dropout_rate eklendi
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_embed, n_head, head_size, dropout_rate) # dropout_rate'i MultiHeadAttention'a geçirme
        self.ffwd = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout_rate) # FFN çıkışına dropout eklendi
        )
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
        self.dropout1 = nn.Dropout(dropout_rate) # Dikkat çıkışı için dropout
        self.dropout2 = nn.Dropout(dropout_rate) # FFN çıkışı için dropout

    def forward(self, x):
        x = x + self.dropout1(self.sa(self.ln1(x))) # Dikkat: norm önce, sonra dropout
        x = x + self.dropout2(self.ffwd(self.ln2(x))) # Dikkat: norm önce, sonra dropout
        return x

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, n_embed: int = 64,
                 n_head: int = 4, n_layer: int = 3, dropout_rate: float = DROPOUT_RATE): # n_embed 32'den 64'e güncellendi
        super().__init__()
        # Her token için bir embedding lookup table (tek başına ne ifade eder)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        # Position embedding table (hangi pozisyonda ne ifade eder)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, n_embed)
        self.tok_pos_dropout = nn.Dropout(dropout_rate) # Embedding dropout eklendi
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head, dropout_rate=dropout_rate) for _ in range(n_layer)]) # dropout_rate'i Block'a geçirme
        self.ln_f = nn.LayerNorm(n_embed) # final layer norm
        self.lm_head = nn.Linear(n_embed, vocab_size) # dil modeli başı

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None):
        B, T = idx.shape

        # idx ve targets tensörleri (B, T) şeklinde, long type
        tok_emb = self.token_embedding_table(idx) # (B,T,C_embed)
        pos_emb = self.position_embedding_table(torch.arange(T, device=DEVICE)) # (T,C_embed)
        x = tok_emb + pos_emb # (B,T,C_embed)
        x = self.tok_pos_dropout(x) # Embedding'lere dropout uygulandı
        x = self.blocks(x) # (B,T,C_embed)
        x = self.ln_f(x) # (B,T,C_embed)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            # PyTorch'un CrossEntropyLoss'u B,T,C_vocab formatında logit beklemiyor.
            # B*T, C_vocab ve B*T şeklinde olmalı.
            B, T, C_vocab = logits.shape
            logits = logits.view(B * T, C_vocab)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx: torch.Tensor, max_new_tokens: int):
        # idx (B, T) şeklinde bir array of indices in the current context
        for _ in range(max_new_tokens):
            # Mevcut blok boyutumuz kadar kırp (çünkü position embedding o kadar)
            idx_cond = idx[:, -BLOCK_SIZE:]
            # Tahminleri al
            logits, loss = self(idx_cond) # self(idx) means calling forward(idx) implicitly
            # Sadece son adıma odaklan
            logits = logits[:, -1, :] # (B, C_vocab)
            # Softmax ile olasılıkları al
            probs = F.softmax(logits, dim=-1) # (B, C_vocab)
            # Bir sonraki token için örnek al
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Oluşan sıraya ekle
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

Next, let's implement the training loop. This will involve instantiating the model, setting up an optimizer, and then iteratively training and evaluating the model.

In [ ]:
# ---------------------------------------------------------------------------
# Eğitim döngüsü ve yardımcı fonksiyonlar
# ---------------------------------------------------------------------------

@torch.no_grad()
def estimate_loss(model, train_data, val_data, encode):
    out = {}
    model.eval() # Model eval moduna geçiyor
    for split in ['train', 'val']:
        losses = torch.zeros(100)
        for k in range(100):
            X, Y = get_batch(split, train_data, val_data, device=DEVICE)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Model train moduna geri dönüyor
    return out

# Veriyi indir ve hazırla
text = download_dataset()
chars, vocab_size, encode, decode = build_tokenizer(text)
train_data, val_data = make_splits(text, encode)

# Modeli oluştur ve device'a taşı
model = BigramLanguageModel(vocab_size).to(DEVICE)

# Optimizer ayarla
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Eğitim döngüsü
EPOCHS = 30000
EVAL_INTERVAL = 500
for iter in range(EPOCHS):
    # Her N adımda bir loss'u değerlendir
    if iter % EVAL_INTERVAL == 0:
        losses = estimate_loss(model, train_data, val_data, encode)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Batch çek
    xb, yb = get_batch('train', train_data, val_data, device=DEVICE)

    # Loss'u hesapla
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Final train loss: {losses['train']:.4f}, final val loss: {losses['val']:.4f}")

Finally, let's use the trained model to generate some text!

In [ ]:
# Modelden metin üret
# Boş bir başlangıç token'ı (yeni satır karakteri) ile başlıyoruz
context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))